# 본문 수집 CSV 통계 (Colab용)

`본문_bs4_*.csv` 파일을 기준으로 월별 수집량, URL 원본 대비 수집률, 결측치, 중복, 카테고리 분포를 확인한다. 한글 파일명 정규화 차이로 파일을 못 찾는 문제를 줄이기 위해 파일명을 정규화해서 처리한다.

- 입력: `data/본문_bs4_*.csv`, `data/링크_*.json`, `data/본문_bs4_재실패_*.json`
- 출력: 월별 본문 수집량, 쿼리별 요약, 수집률, 결측치, 카테고리 분포
- 저장 파일: `통계_본문수집_월별.csv`, `통계_본문수집_pivot.csv`, `통계_본문수집_query요약.csv`, `통계_본문수집_카테고리.csv`, `통계_본문수집_결측치.csv`


In [12]:
# Colab에서 실행할 때만 사용
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
from pathlib import Path
import json
import os
import re
import unicodedata

import pandas as pd

# Colab 기본 경로. 로컬/WSL에서 실행하면 아래 except 경로를 사용
try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists():
        raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')

os.chdir(PROJECT_DIR)  # 상대경로가 프로젝트 기준으로 잡히도록 작업 폴더 변경
print(f'현재 작업 폴더: {Path.cwd()}')

DATA_DIR = PROJECT_DIR / 'notebook' / 'crawling' / 'data'
print(f'DATA_DIR: {DATA_DIR}')


현재 작업 폴더: /content/drive/MyDrive/Text-data-Analysis_26-Spring
DATA_DIR: /content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data


In [14]:
def normalize_text(text):
    return unicodedata.normalize('NFC', str(text))


def normalize_name(path):
    return normalize_text(path.name)


def list_files_normalized(data_dir=DATA_DIR):
    files = []
    for path in data_dir.iterdir():
        if path.is_file():
            files.append((path, normalize_name(path)))
    return files


DATA_FILES = list_files_normalized()
print(f'DATA_DIR 파일 수: {len(DATA_FILES)}')
print('본문 CSV 후보:', sum(name.startswith('본문_bs4_') and name.endswith('.csv') for _, name in DATA_FILES))


def find_file_by_normalized_name(target_name):
    target_name = normalize_text(target_name)
    for path, name in DATA_FILES:
        if name == target_name:
            return path
    return None


def parse_body_filename(path):
    name = normalize_name(path)
    match = re.match(r'^본문_bs4_(.+)_(\d{6})_(\d{6})\.csv$', name)
    if not match:
        return None

    query, start_ymd, end_ymd = match.groups()
    return {
        'query': query,
        'period': f'{start_ymd}_{end_ymd}',
        'start_ym': f'20{start_ymd[:2]}.{start_ymd[2:4]}',
        'start_ymd': start_ymd,
        'end_ymd': end_ymd,
    }


def read_url_count(query, period):
    link_path = find_file_by_normalized_name(f'링크_{query}_{period}.json')
    if link_path is None:
        return None
    with link_path.open('r', encoding='utf-8') as f:
        return len(json.load(f))


def read_fail_count(query, period):
    fail_path = find_file_by_normalized_name(f'본문_bs4_재실패_{query}_{period}.json')
    if fail_path is None:
        return 0
    with fail_path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    return len(data.get('err_idx', []))


DATA_DIR 파일 수: 75
본문 CSV 후보: 25


In [15]:
summary_rows = []
category_rows = []
missing_rows = []

body_files = sorted(
    [path for path, name in DATA_FILES if name.startswith('본문_bs4_') and name.endswith('.csv') and parse_body_filename(path)],
    key=lambda path: normalize_name(path),
)

if not body_files:
    sample_names = [name for _, name in DATA_FILES[:20]]
    raise ValueError(f'본문_bs4_*.csv 파일을 찾지 못했습니다. DATA_DIR 확인 필요: {DATA_DIR}\n샘플 파일명: {sample_names}')

for csv_path in body_files:
    info = parse_body_filename(csv_path)
    df = pd.read_csv(csv_path, encoding='utf-8-sig')

    url_count = read_url_count(info['query'], info['period'])
    fail_count = read_fail_count(info['query'], info['period'])
    row_count = len(df)
    unique_links = df['link'].nunique() if 'link' in df.columns else None
    duplicate_links = row_count - unique_links if unique_links is not None else None

    summary_rows.append({
        **info,
        'csv_rows': row_count,
        'unique_links': unique_links,
        'duplicate_links': duplicate_links,
        'url_count': url_count,
        'failed_count': fail_count,
        'missing_vs_url': None if url_count is None else url_count - row_count,
        'collection_rate': None if not url_count else row_count / url_count,
        'file_mb': csv_path.stat().st_size / 1024 / 1024,
    })

    for col in ['link', 'pubdate', 'category', 'title', 'body']:
        if col in df.columns:
            missing_rows.append({
                **info,
                'column': col,
                'missing_count': int(df[col].isna().sum() + (df[col].astype(str).str.strip() == '').sum()),
            })

    if 'category' in df.columns:
        counts = df['category'].fillna('').replace('', '(비어있음)').value_counts()
        for category, count in counts.items():
            category_rows.append({**info, 'category': category, 'count': int(count)})

summary_df = pd.DataFrame(summary_rows).sort_values(['query', 'start_ym']).reset_index(drop=True)
category_df = pd.DataFrame(category_rows).sort_values(['query', 'start_ym', 'count'], ascending=[True, True, False]).reset_index(drop=True)
missing_df = pd.DataFrame(missing_rows).sort_values(['query', 'start_ym', 'column']).reset_index(drop=True)

summary_df


,query,period,start_ym,start_ymd,end_ymd,csv_rows,unique_links,duplicate_links,url_count,failed_count,missing_vs_url,collection_rate,file_mb
0,KT,250901_250930,2025.09,250901,250930,7878,7878,0,7900,22,22,0.997215,20.667343
1,KT,251001_251031,2025.10,251001,251031,3686,3686,0,3688,2,2,0.999458,11.135546
2,KT,251101_251130,2025.11,251101,251130,4439,4439,0,4442,3,3,0.999325,11.393623
3,KT,251201_251231,2025.12,251201,251231,5008,5008,0,5018,10,10,0.998007,14.911847
4,KT,260101_260131,2026.01,260101,260131,2856,2856,0,2859,3,3,0.998951,8.315452
5,LG U+,250801_250831,2025.08,250801,250831,478,478,0,478,0,0,1.000000,1.355374
6,LG U+,250901_250930,2025.09,250901,250930,556,556,0,557,1,1,0.998205,1.478086
7,LG U+,251001_251031,2025.10,251001,251031,534,534,0,534,0,0,1.000000,1.362261
8,LG U+,251101_251130,2025.11,251101,251130,567,567,0,567,0,0,1.000000,1.626827
9,LG U+,251201_251231,2025.12,251201,251231,645,645,0,645,0,0,1.000000,1.737975


In [16]:
# 쿼리 x 월별 본문 수집 건수
body_count_pivot = summary_df.pivot_table(
    index='query',
    columns='start_ym',
    values='csv_rows',
    aggfunc='sum',
    fill_value=0,
)
body_count_pivot['total'] = body_count_pivot.sum(axis=1)
body_count_pivot


start_ym,2025.04,2025.05,2025.06,2025.07,2025.08,2025.09,2025.10,2025.11,2025.12,2026.01,total
query,,,,,,,,,,,
KT,0,0,0,0,0,7878,3686,4439,5008,2856,23867
LG U+,0,0,0,0,478,556,534,567,645,0,2780
LG유플러스,0,0,0,0,1400,1787,1428,1482,1435,0,7532
SKT,4454,5688,1473,2090,1787,0,0,0,0,0,15492
SK텔레콤,5361,6474,2384,2993,2810,0,0,0,0,0,20022


In [17]:
# URL 원본 대비 수집률
rate_pivot = summary_df.pivot_table(
    index='query',
    columns='start_ym',
    values='collection_rate',
    aggfunc='mean',
)
(rate_pivot * 100).round(2)


start_ym,2025.04,2025.05,2025.06,2025.07,2025.08,2025.09,2025.10,2025.11,2025.12,2026.01
query,,,,,,,,,,
KT,NaN,NaN,NaN,NaN,NaN,99.72,99.95,99.93,99.8,99.9
LG U+,NaN,NaN,NaN,NaN,100.00,99.82,100.00,100.00,100.0,NaN
LG유플러스,NaN,NaN,NaN,NaN,100.00,100.00,100.00,100.00,100.0,NaN
SKT,100.0,99.86,100.0,99.48,99.94,NaN,NaN,NaN,NaN,NaN
SK텔레콤,100.0,100.00,100.0,100.00,100.00,NaN,NaN,NaN,NaN,NaN


In [18]:
# 실패/누락이 있는 파일만 확인
summary_df[
    (summary_df['failed_count'] > 0) |
    (summary_df['missing_vs_url'] > 0) |
    (summary_df['duplicate_links'] > 0)
].sort_values(['failed_count', 'missing_vs_url'], ascending=False)


,query,period,start_ym,start_ymd,end_ymd,csv_rows,unique_links,duplicate_links,url_count,failed_count,missing_vs_url,collection_rate,file_mb
0,KT,250901_250930,2025.09,250901,250930,7878,7878,0,7900,22,22,0.997215,20.667343
18,SKT,250701_250731,2025.07,250701,250731,2090,2090,0,2101,11,11,0.994764,6.303063
3,KT,251201_251231,2025.12,251201,251231,5008,5008,0,5018,10,10,0.998007,14.911847
16,SKT,250501_250531,2025.05,250501,250531,5688,5688,0,5696,8,8,0.998596,14.919526
2,KT,251101_251130,2025.11,251101,251130,4439,4439,0,4442,3,3,0.999325,11.393623
4,KT,260101_260131,2026.01,260101,260131,2856,2856,0,2859,3,3,0.998951,8.315452
1,KT,251001_251031,2025.10,251001,251031,3686,3686,0,3688,2,2,0.999458,11.135546
6,LG U+,250901_250930,2025.09,250901,250930,556,556,0,557,1,1,0.998205,1.478086
19,SKT,250801_250831,2025.08,250801,250831,1787,1787,0,1788,1,1,0.999441,5.687033


In [19]:
# 쿼리별 요약
query_summary = summary_df.groupby('query', as_index=False).agg(
    csv_rows=('csv_rows', 'sum'),
    unique_links=('unique_links', 'sum'),
    url_count=('url_count', 'sum'),
    failed_count=('failed_count', 'sum'),
    duplicate_links=('duplicate_links', 'sum'),
    file_mb=('file_mb', 'sum'),
)
query_summary['missing_vs_url'] = query_summary['url_count'] - query_summary['csv_rows']
query_summary['collection_rate'] = query_summary['csv_rows'] / query_summary['url_count']
query_summary.sort_values('csv_rows', ascending=False).reset_index(drop=True)


,query,csv_rows,unique_links,url_count,failed_count,duplicate_links,file_mb,missing_vs_url,collection_rate
0,KT,23867,23867,23907,40,0,66.423811,40,0.998327
1,SK텔레콤,20022,20022,20022,0,0,57.338216,0,1.000000
2,SKT,15492,15492,15512,20,0,42.064286,20,0.998711
3,LG유플러스,7532,7532,7532,0,0,23.740289,0,1.000000
4,LG U+,2780,2780,2781,1,0,7.560524,1,0.999640


In [20]:
# 전체 카테고리 분포
category_total = category_df.groupby(['query', 'category'], as_index=False)['count'].sum()
category_total.sort_values(['query', 'count'], ascending=[True, False]).reset_index(drop=True)


,query,category,count
0,KT,IT/과학,8537
1,KT,사회,7167
2,KT,경제,5280
3,KT,정치,1630
4,KT,세계,441
5,KT,생활/문화,402
6,KT,오피니언,223
7,KT,(비어있음),187
8,LG U+,IT/과학,2050
9,LG U+,경제,436


In [21]:
# 결측치 확인
missing_df[missing_df['missing_count'] > 0].reset_index(drop=True)


,query,period,start_ym,start_ymd,end_ymd,column,missing_count
0,KT,250901_250930,2025.09,250901,250930,category,63
1,KT,251001_251031,2025.10,251001,251031,category,29
2,KT,251101_251130,2025.11,251101,251130,category,37
3,KT,251201_251231,2025.12,251201,251231,category,34
4,KT,260101_260131,2026.01,260101,260131,category,24
5,LG U+,250801_250831,2025.08,250801,250831,category,1
6,LG U+,250901_250930,2025.09,250901,250930,category,1
7,LG U+,251001_251031,2025.10,251001,251031,category,6
8,LG U+,251101_251130,2025.11,251101,251130,category,1
9,LG U+,251201_251231,2025.12,251201,251231,category,2


In [22]:
# 통계 결과 저장
summary_path = DATA_DIR / '통계_본문수집_월별.csv'
pivot_path = DATA_DIR / '통계_본문수집_pivot.csv'
query_summary_path = DATA_DIR / '통계_본문수집_query요약.csv'
category_path = DATA_DIR / '통계_본문수집_카테고리.csv'
missing_path = DATA_DIR / '통계_본문수집_결측치.csv'

summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
body_count_pivot.to_csv(pivot_path, encoding='utf-8-sig')
query_summary.to_csv(query_summary_path, index=False, encoding='utf-8-sig')
category_df.to_csv(category_path, index=False, encoding='utf-8-sig')
missing_df.to_csv(missing_path, index=False, encoding='utf-8-sig')

print(summary_path)
print(pivot_path)
print(query_summary_path)
print(category_path)
print(missing_path)


/content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data/통계_본문수집_월별.csv
/content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data/통계_본문수집_pivot.csv
/content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data/통계_본문수집_query요약.csv
/content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data/통계_본문수집_카테고리.csv
/content/drive/MyDrive/Text-data-Analysis_26-Spring/notebook/crawling/data/통계_본문수집_결측치.csv


## 결과 해석

전체 본문 수집 결과는 원본 URL 69,754개 중 69,693개가 CSV로 저장되어, 전체 수집률은 약 99.91%이다. 실패한 61건은 전체의 약 0.09%로 매우 작은 비율이며, 확인 결과 대부분 본문 영역이 없는 속보성 기사라 분석 대상에서 제외해도 전체 결과에 미치는 영향은 제한적이다.

쿼리별로 보면 `SK텔레콤`과 `LG유플러스`는 URL 수와 CSV 행 수가 동일해 100% 수집되었다. `KT`는 23,907개 중 23,867개, `SKT`는 15,512개 중 15,492개, `LG U+`는 2,781개 중 2,780개가 수집되어 일부 실패가 있었지만 모두 99% 이상 수집률을 보인다.

실패가 상대적으로 많았던 구간은 `KT 2025.09` 22건, `SKT 2025.07` 11건, `KT 2025.12` 10건이다. 다만 월별 최저 수집률도 `SKT 2025.07`의 약 99.48% 수준이라, 특정 월의 데이터가 크게 훼손된 정도는 아니다.

카테고리는 `IT/과학`이 가장 많고, 그다음 `경제`, `사회`, `정치` 순으로 나타난다. 통신사 관련 키워드 수집이라는 점을 고려하면 `IT/과학`과 `경제` 비중이 큰 것은 자연스러운 분포로 볼 수 있다. 카테고리 결측은 566건이며, 본문/제목/날짜 결측은 별도로 관찰되지 않아 본문 분석에는 큰 문제가 없어 보인다.

따라서 현재 본문 CSV는 전체 분석에 사용하기 충분한 품질로 보이며, 실패 URL은 “본문이 없는 속보성 기사 등으로 인해 제외된 사례”로 정리하면 된다.
